In [2]:
import os
import torch
from transformers import RobertaTokenizerFast, RobertaModel
from data import build_spacy_and_token_cache, create_dataloader, loadi2b2
from train import train_model
from model import create_model
from evaluate import evaluate_model, evaluate_closure

In [5]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")
special_tokens = {"additional_special_tokens": ["[E1]", "[/E1]", "[E2]", "[/E2]"]}
num_added_toks = tokenizer.add_special_tokens(special_tokens)
bert.resize_token_embeddings(len(tokenizer))

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Embedding(30526, 768, padding_idx=0)

# Load Data

In [9]:
data_info = loadi2b2()

train_df = data_info['train_df']
valid_df = data_info['valid_df']
test_df = data_info['test_df']
num_classes = data_info['num_classes']
relation_map = data_info['label_mapping']
id2label = data_info['id2label']

type_map = {"UNKNOWN": 0, "OCCURRENCE": 1, "TREATMENT": 2, "TEST": 3, "DURATION": 4, "PROBLEM": 5, "CLINICAL_DEPT": 6, "EVIDENTIAL": 7,
            "DATE": 8, "FREQUENCY": 9, "TIME": 10,
        }
num_types = 11
rule_filename = 'rules_i2b2.txt'

cache_path = '/data/ddao/TRE/pretrained_models/Reasoning/i2b2/cache/'

os.makedirs(cache_path + "train/", exist_ok=True)
os.makedirs(cache_path + "test/", exist_ok=True)
os.makedirs(cache_path + "valid/", exist_ok=True)

train_token_cache_path = cache_path + "train/token_cache_bert.pt"
train_spacy_jsonl_path = cache_path + "train/spacy_cache_bert.jsonl"

test_token_cache_path = cache_path + "test/token_cache_bert.pt"
test_spacy_jsonl_path = cache_path + "test/spacy_cache_bert.jsonl"

valid_token_cache_path = cache_path + "valid/token_cache_bert.pt"
valid_spacy_jsonl_path = cache_path + "valid/spacy_cache_bert.jsonl"

batch_size = 128
window_size = 128
MAX_LENGTH = 256

In [10]:
train_loader = create_dataloader(train_token_cache_path, train_spacy_jsonl_path, batch_size=batch_size)
val_loader   = create_dataloader(valid_token_cache_path, valid_spacy_jsonl_path, batch_size=batch_size)
test_loader  = create_dataloader(test_token_cache_path, test_spacy_jsonl_path, batch_size=batch_size)

# Create Model

In [11]:
mode           = 'TRER'
enc_layers     = 2
nhead          = 8
p_drop         = 0.1
step_size      = 0.01
lambda_kl      = 0.1
smooth_tau     = 0.1
lambda_entropy = 0.1
max_steps      = 100
epochs         = 40
lr             = 0.00001
lambda_distill = 0.1
use_amp        = True
freeze_bert    = True
save_path      = f"/data/ddao/TRE/pretrained_models/Reasoning/i2b2/{mode}_RoBERTa.pt"

In [12]:
model = create_model(
    mode=mode,
    bert=bert,
    num_classes=num_classes,
    num_types=num_types,
    type_map=type_map,
    relation_map=relation_map,
    enc_layers=enc_layers,
    nhead=nhead,
    p_drop=p_drop,
    step_size=step_size,
    lambda_kl=lambda_kl,
    smooth_tau=smooth_tau,
    lambda_entropy=lambda_entropy,
    max_steps=max_steps,
    rule_file_path=rule_filename,
    freeze_bert=freeze_bert
)

[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_BO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_AO
[UnifiedNeuralReasoningLayer] Hard Rule: R_Trans_OA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_BA
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_AB
[UnifiedNeuralReasoningLayer] Hard Rule: R_Symm_OO
[UnifiedNeuralReasoningLayer] init: rules=16, rel=3, types=11, smooth_tau=0.1, lambda_kl=0.1, lambda_entropy=0.1


# Train

In [ ]:
train_model(model,
            train_loader,
            val_loader,
            epochs,
            lr,
            lambda_distill,
            save_path,
            use_amp=True)

Using device: cuda


Evaluate: 100%|██████████| 40/40 [00:25<00:00,  1.56it/s]


Epoch 0: Loss=0.6683 (CE=0.6683, KL=0.0000, Energy=0.0841) | Val F1=0.7577
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/BERT.pt (F1: 0.7577)


Evaluate: 100%|██████████| 40/40 [00:25<00:00,  1.57it/s]


Epoch 1: Loss=0.5736 (CE=0.5736, KL=0.0000, Energy=0.1014) | Val F1=0.7760
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/BERT.pt (F1: 0.7760)


...

Evaluate: 100%|██████████| 40/40 [00:25<00:00,  1.57it/s]


Epoch 15: Loss=0.3614 (CE=0.3614, KL=-0.0000, Energy=0.1326) | Val F1=0.8393
  Saved best model to /data/ddao/TRE/pretrained_models/Reasoning/i2b2/BERT.pt (F1: 0.8393)


Evaluate: 100%|██████████| 40/40 [00:25<00:00,  1.57it/s]


Epoch 16: Loss=0.3466 (CE=0.3466, KL=-0.0000, Energy=0.1343) | Val F1=0.8347


Epoch 17 (Reasoning):  86%|████████▋ | 192/222 [00:51<00:08,  3.75it/s]

# Evaluate

In [19]:
print("Load pretrained model:")
checkpoint   = torch.load(save_path)
eval_model   = model.load_state_dict(checkpoint['model_state_dict'])

Load pretrained model:


<All keys matched successfully>

In [21]:
cm, f1, p, r = evaluate_model(
    test_loader=test_loader,
    model=eval_model
)

Evaluate: 100%|██████████| 216/216 [02:39<00:00,  1.36it/s]


In [22]:
print('CM:\n', cm)
print('f1:', f1, 'p', p, 'r', r)

CM:
 [[ 1747   403   575]
 [  330 13548  1039]
 [  475   717  8696]]
f1: 0.8710719984716709 p 0.8711761391143793 r 0.8714493280058119
